In [1]:
from pathlib import Path
from prettyprinter import cpprint
import json
from resistics.resp import load

proj_dir = Path.home() / "magnetotellurics" / "projects" / "resistics_mth5"
proj = load(proj_dir)
cpprint(json.loads(proj.model_dump_json()))


2024-02-23T16:11:49.651147+0000 | INFO | mth5.mth5 | filename | file extension .h5 is not correct. Changing to default .h5
{
    'dir_path': '/home/ringo_dingo/magnetotellurics/projects/resistics_mth5',
    'mth5_path': '/home/ringo_dingo/magnetotellurics/mth5_data/8P_CAS04_NVR08.h5',
    'ref_time': '2020-06-02 00:00:00.000000_000000_000000_000000',
    'surveys': ['CONUS South'],
    'stations': ['CONUS South/CAS04', 'CONUS South/NVR08'],
    'runs': [
        'CONUS South/CAS04/a',
        'CONUS South/CAS04/b',
        'CONUS South/CAS04/c',
        'CONUS South/CAS04/d',
        'CONUS South/NVR08/a',
        'CONUS South/NVR08/b',
        'CONUS South/NVR08/c'
    ]
}


We want to process the station CAS04 in two ways:

- single site processing
- remote reference processing

The steps in the processing are the following:

- read the time data
- preprocess the time data
- window and decimate
- convert to frequency domain
- calculate out evaluation frequencies
- do the regression



Begin by getting all the runs we need to process

In [2]:
runs = proj.get_runs(survey="CONUS South", station="CAS04", fs=1)
runs


{'CONUS South/CAS04/a': /Experiment/Surveys/CONUS_South/Stations/CAS04/a:
     --> Dataset: ex
     .................
     --> Dataset: ey
     .................
     --> Dataset: hx
     .................
     --> Dataset: hy
     .................
     --> Dataset: hz
     .................,
 'CONUS South/CAS04/b': /Experiment/Surveys/CONUS_South/Stations/CAS04/b:
     --> Dataset: ex
     .................
     --> Dataset: ey
     .................
     --> Dataset: hx
     .................
     --> Dataset: hy
     .................
     --> Dataset: hz
     .................,
 'CONUS South/CAS04/c': /Experiment/Surveys/CONUS_South/Stations/CAS04/c:
     --> Dataset: ex
     .................
     --> Dataset: ey
     .................
     --> Dataset: hx
     .................
     --> Dataset: hy
     .................
     --> Dataset: hz
     .................,
 'CONUS South/CAS04/d': /Experiment/Surveys/CONUS_South/Stations/CAS04/d:
     --> Dataset: ex
     ...............

Loop over the runs and get the time data

In [10]:
for run_name, run in runs.items():
    time_data = run.to_runts()
    time_data = time_data.calibrate()
